# Chapter 04 — Stateful Processing, Timers & Hot Key Fanout

Welcome to **Chapter 04** of the **Google Cloud Dataflow Workshop**!

In this chapter, you will master two advanced distributed stream processing patterns:
1. **The Hot Key Problem & Fanout**: Mitigating severe data skew (`with_hot_key_fanout`) when a single entity receives orders of magnitude more traffic than others.
2. **Stateful Stream Processing (`@StateId`)**: Maintaining per-key rolling statistics, counters, and visited histories in local state without querying external databases.
3. **State TTL & Timers (`@TimerId`)**: Scheduling watermark-based timers to automatically flush sessions and clear in-memory state to prevent memory leaks in 24/7 pipelines.

## 1. Environment Verification & Kernel Selection

Ensure that your notebook kernel is attached to the project's **`.venv`** managed by **`uv`**:
- Check the top-right kernel selector and select **`Python (dataflow-workshops)`**.

Run the cell below to verify your environment:

In [1]:
import os
import sys

print(f"Python Executable: {sys.executable}")
if ".venv" in sys.executable:
    print("✅ Success: Notebook is running inside the project's .venv!")
else:
    print("⚠️ Warning: Notebook is running on system Python! Select 'Python (dataflow-workshops)' from the top-right kernel picker.")

import apache_beam as beam
print(f"✅ Apache Beam Version: {beam.__version__}")

Python Executable: /Users/zelda.luconi/Medium/google-products/dataflow-workshops/.venv/bin/python3
✅ Success: Notebook is running inside the project's .venv!


✅ Apache Beam Version: 2.77.0


## 2. Architecture & Visual DAG

Here is the architecture for this chapter. Notice the two distinct engines: the **Stateful & Timer Machine** and the **Hot Key Fanout Engine**:

![Chapter 04 Stateful & Fanout DAG](../artifacts/04_stateful_and_fanout_dag.png)

<details>
<summary><b>Click to expand Mermaid source code</b></summary>

```mermaid
flowchart TD
    classDef pcoll fill:#e8f0fe,stroke:#4285f4,stroke-width:2px,color:#1a73e8,font-weight:bold;
    classDef ptrans fill:#fef7e0,stroke:#f9ab00,stroke-width:2px,color:#b06000,font-weight:bold;
    classDef sink fill:#e6f4ea,stroke:#34a853,stroke-width:2px,color:#137333,font-weight:bold;
    classDef state fill:#f3e8fd,stroke:#9334e6,stroke-width:2px,color:#6b21a8,font-weight:bold;
    classDef hotkey fill:#fee2e2,stroke:#ef4444,stroke-width:2px,color:#991b1b,font-weight:bold;

    INPUT[skewed_clickstream.json]:::hotkey --> READ[ReadClickstream]:::ptrans
    READ --> PARSE[ParseAndTimestamp]:::ptrans
    PARSE --> STATEFUL[StatefulSessionTracker]:::state
    STATEFUL --> SINK_STATE[BigQuery user_session_profiles]:::sink
    PARSE --> FANOUT[CombineWithFanout with_hot_key_fanout]:::hotkey
    FANOUT --> SINK_FAN[BigQuery fanout_user_metrics]:::sink
```
</details>

## 3. Inspecting Data Skew in `skewed_clickstream.json`

Let's load the dataset and compute the event count per user to visualize the severe data skew:

In [2]:
from collections import Counter
import json

user_counts = Counter()
with open("data/skewed_clickstream.json") as f:
    for line in f:
        line = line.strip()
        if line:
            item = json.loads(line)
            user_counts[item["user_id"]] += 1

print("=== Event Distribution Across Users ===")
for user, count in user_counts.most_common():
    bar = "█" * (count // 15)
    print(f"{user:<22} : {count:>4} events | {bar}")

=== Event Distribution Across Users ===
usr_hot_influencer     :  500 events | █████████████████████████████████
usr_carol              :    5 events | 
usr_elena              :    5 events | 
usr_bob                :    5 events | 
usr_alice              :    5 events | 
usr_david              :    5 events | 


## 4. Concept 1: Mitigating Hot Keys with `with_hot_key_fanout`

When performing a `GroupByKey` or `CombinePerKey` on skewed keys:
- In standard Beam/Dataflow, **all elements of a given key route to a single worker**.
- Adding 100 or 1,000 workers does **not** solve the bottleneck; the worker holding `usr_hot_influencer` crashes with out-of-memory errors!

By using `.with_hot_key_fanout(fanout=10)`, Beam automatically splits the hot key across 10 intermediate workers using random salts, aggregates partial sums locally, and merges the partial results globally:

In [3]:
import apache_beam as beam

sample_skewed_data = [
    ("hot_influencer", 10.0),
    ("hot_influencer", 20.0),
    ("hot_influencer", 30.0),
    ("hot_influencer", 40.0),
    ("regular_alice",  50.0),
]

with beam.Pipeline() as p:
    (
        p
        | "CreateSkewed" >> beam.Create(sample_skewed_data)
        | "CombineWithFanout" >> beam.CombinePerKey(sum).with_hot_key_fanout(
            lambda key: 5 if "hot" in key else 0
        )
        | "PrintResults" >> beam.Map(lambda kv: print(f"Key: {kv[0]:<16} | Aggregated Sum: ${kv[1]:.2f}"))
    )

Key: regular_alice    | Aggregated Sum: $50.00
Key: hot_influencer   | Aggregated Sum: $100.00


## 5. Concept 2: Stateful Stream Processing with `@StateId`

Beam's Stateful Processing API allows a `DoFn` to maintain per-key state in local memory and SSD cache (managed by Dataflow Streaming Engine):
- `ReadModifyWriteStateSpec`: Single scalar values (counters, rolling sums, timestamps).
- `BagStateSpec`: Append-only collection of elements (e.g. visited pages history).

Let's test maintaining state across records:

In [4]:
from apache_beam.transforms.userstate import ReadModifyWriteStateSpec, BagStateSpec
from apache_beam.coders import VarIntCoder, StrUtf8Coder

class StatefulAccumulatorFn(beam.DoFn):
    COUNT_STATE = ReadModifyWriteStateSpec("count", VarIntCoder())
    PAGES_STATE = BagStateSpec("pages", StrUtf8Coder())

    def process(
        self,
        element,
        count_state=beam.DoFn.StateParam(COUNT_STATE),
        pages_state=beam.DoFn.StateParam(PAGES_STATE),
    ):
        user, page = element
        current_count = (count_state.read() or 0) + 1
        count_state.write(current_count)
        pages_state.add(page)

        yield (user, current_count, list(pages_state.read()))

events = [
    ("alice", "/home"),
    ("bob", "/shop"),
    ("alice", "/product/101"),
    ("alice", "/cart"),
]

with beam.Pipeline() as p:
    (
        p
        | beam.Create(events)
        | beam.ParDo(StatefulAccumulatorFn())
        | beam.Map(lambda res: print(f"User: {res[0]:<6} | Event Count: {res[1]} | Visited Pages: {res[2]}"))
    )

User: alice  | Event Count: 1 | Visited Pages: ['/home']
User: bob    | Event Count: 1 | Visited Pages: ['/shop']
User: alice  | Event Count: 2 | Visited Pages: ['/home', '/product/101']
User: alice  | Event Count: 3 | Visited Pages: ['/home', '/product/101', '/cart']


## 6. Concept 3: Automated State Expiration with `@TimerId`

Without garbage collection, state stored in a 24/7 streaming pipeline leaks memory until workers run out of memory.

Using `@TimerId` with `TimeDomain.WATERMARK`, we schedule an inactivity timer whenever an element arrives.
When the watermark advances past the scheduled timestamp, `@on_timer` fires, emits the finalized profile, and calls `state.clear()` to reclaim memory:

In [5]:
from apache_beam.transforms.userstate import TimerSpec, on_timer
from apache_beam import TimeDomain

class StatefulTimerSessionFn(beam.DoFn):
    COUNT_STATE = ReadModifyWriteStateSpec("count", VarIntCoder())
    EXPIRY_TIMER = TimerSpec("expiry", TimeDomain.WATERMARK)

    def process(
        self,
        element,
        ts=beam.DoFn.TimestampParam,
        count_state=beam.DoFn.StateParam(COUNT_STATE),
        timer=beam.DoFn.TimerParam(EXPIRY_TIMER),
    ):
        user, _ = element
        count = (count_state.read() or 0) + 1
        count_state.write(count)
        # Set timer 5 seconds past element timestamp
        timer.set(ts + 5)
        yield ("EVENT_PROCESSED", user, count)

    @on_timer(EXPIRY_TIMER)
    def on_expiry(
        self,
        key=beam.DoFn.KeyParam,
        count_state=beam.DoFn.StateParam(COUNT_STATE),
    ):
        final_count = count_state.read() or 0
        count_state.clear()  # Reclaim state memory!
        yield ("SESSION_FINALIZED_ON_TIMER", key, final_count)

with beam.Pipeline() as p:
    (
        p
        | beam.Create([("user_alpha", "evt_1"), ("user_alpha", "evt_2")])
        | beam.ParDo(StatefulTimerSessionFn())
        | beam.Map(print)
    )

('EVENT_PROCESSED', 'user_alpha', 1)
('EVENT_PROCESSED', 'user_alpha', 2)
('SESSION_FINALIZED_ON_TIMER', 'user_alpha', 2)


## 7. Running the Full Production Pipeline Locally

Now let's execute the complete pipeline script [`pipelines/stateful_and_fanout_pipeline.py`](pipelines/stateful_and_fanout_pipeline.py) using `DirectRunner`.

This will process the entire `data/skewed_clickstream.json` dataset across both branches and write outputs to `output/`:

In [6]:
import subprocess

cmd = ["bash", "scripts/01_run_local.sh"]
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)

 Chapter 04: Running Stateful Processing & Hot Key Fanout Pipeline
Input File:      /Users/zelda.luconi/Medium/google-products/dataflow-workshops/04_stateful_processing_and_hotkeys/data/skewed_clickstream.json
Stateful Output: /Users/zelda.luconi/Medium/google-products/dataflow-workshops/04_stateful_processing_and_hotkeys/output/stateful_sessions*.json
Fanout Output:   /Users/zelda.luconi/Medium/google-products/dataflow-workshops/04_stateful_processing_and_hotkeys/output/fanout_metrics*.json
----------------------------------------------------------------------
----------------------------------------------------------------------
 Execution finished! Output files in /Users/zelda.luconi/Medium/google-products/dataflow-workshops/04_stateful_processing_and_hotkeys/output:
total 16
drwxr-xr-x@ 4 zelda.luconi  staff   128 Oct 27 19:54 .
drwxr-xr-x@ 8 zelda.luconi  staff   256 Oct 27 19:54 ..
-rw-r--r--@ 1 zelda.luconi  staff   704 Oct 27 19:54 fanout_metrics-00000-of-00001.json
-rw-r--r--@

## 8. Inspecting Local Output Files

Let's inspect the JSON lines emitted by the pipeline for both stateful sessions and fanout metrics:

In [7]:
import glob
import json

def show_json_lines(pattern, title):
    print(f"=== {title} ===")
    files = glob.glob(pattern)
    if not files:
        print("No files found for pattern:", pattern)
        return
    for f in sorted(files):
        with open(f, "r") as fp:
            for line in fp.readlines()[:6]:
                item = json.loads(line.strip())
                print(json.dumps(item, indent=2))
    print()

show_json_lines("output/stateful_sessions*.json", "Finalized Stateful User Sessions")
show_json_lines("output/fanout_metrics*.json", "Hot Key Fanout Combined Metrics")

=== Finalized Stateful User Sessions ===
{
  "user_id": "usr_elena",
  "first_seen": "2026-10-27T12:00:06+00:00",
  "last_seen": "2026-10-27T12:02:45+00:00",
  "session_duration_sec": 159,
  "total_events": 5,
  "total_spend": 152.05,
  "distinct_pages_visited": 3,
  "is_anomaly_velocity": false,
  "finalized_at": "2026-10-27T18:54:56.838931+00:00"
}
{
  "user_id": "usr_bob",
  "first_seen": "2026-10-27T12:00:08+00:00",
  "last_seen": "2026-10-27T12:02:41+00:00",
  "session_duration_sec": 153,
  "total_events": 5,
  "total_spend": 0.0,
  "distinct_pages_visited": 3,
  "is_anomaly_velocity": false,
  "finalized_at": "2026-10-27T18:54:56.841869+00:00"
}
{
  "user_id": "usr_alice",
  "first_seen": "2026-10-27T12:00:09+00:00",
  "last_seen": "2026-10-27T12:02:51+00:00",
  "session_duration_sec": 162,
  "total_events": 5,
  "total_spend": 187.64,
  "distinct_pages_visited": 3,
  "is_anomaly_velocity": false,
  "finalized_at": "2026-10-27T18:54:56.843791+00:00"
}
{
  "user_id": "usr_david",


## 9. Summary & Next Steps

### What You Learned in Chapter 04:
1. **The Hot Key Problem**: Why adding more worker VMs fails to distribute skewed keys, and how `with_hot_key_fanout` performs automated 2-phase intermediate combining.
2. **Stateful DoFns**: Using `ReadModifyWriteStateSpec` and `BagStateSpec` to maintain per-key counters, sums, and history without external databases.
3. **State TTL & Timers**: Scheduling `@TimerId` to finalize sessions and clear state, preventing memory leaks in 24/7 streaming pipelines.

### Deploying to Google Cloud Dataflow:
To deploy to GCP with **DataflowRunner** and **BigQuery Storage Write API**:

```bash
bash scripts/02_run_dataflow.sh
```

In **Chapter 05**, we will explore **Production Performance, Streaming Engine & Observability** (Dataflow Shuffle, Streaming Engine, System Lag vs. Data Freshness, and Cloud Monitoring SLA alerts).